<a href="https://colab.research.google.com/github/ranggasaviory/Tugas-Praktikum-1-AI-Rangga/blob/main/Tugas_ANN_Predik_Seleb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import numpy as np
import pandas as pd

# Step 1: Loading and Preprocessing Dataset
def load_and_preprocess_data(file_path):
    # Membaca file CSV
    df = pd.read_csv(file_path)

    # Mengambil 6 data pertama seperti permintaan sebelumnya (atau bisa semua data)
    # Kita gunakan fitur: gender, known_for_department, dan popularity
    df = df.head(100) # Kita ambil 100 data agar training lebih terasa

    # Konversi Department (Teks) ke Angka
    df['dept_code'] = pd.Categorical(df['known_for_department']).codes

    # Normalisasi Popularity (Target & Input)
    # Kita buat Target: Jika popularity > median maka 1 (Populer), else 0
    median_pop = df['popularity'].median()
    df['target'] = (df['popularity'] > median_pop).astype(int)

    # Input X: Gender, Dept_Code, Popularity (Normalisasi)
    X = df[['gender', 'dept_code', 'popularity']].values.T
    X[2] = X[2] / np.max(X[2]) # Normalisasi kolom popularity agar range 0-1

    # Target Y
    Y = df['target'].values.reshape(1, -1)

    return X, Y

# Step 2: Initializing the Neural Network
def initialize_parameters(input_size, hidden_size, output_size):
    np.random.seed(42)
    parameters = {
        "W1": np.random.randn(hidden_size, input_size) * 0.01,
        "b1": np.zeros((hidden_size, 1)),
        "W2": np.random.randn(output_size, hidden_size) * 0.01,
        "b2": np.zeros((output_size, 1))
    }
    return parameters

# Step 3: Defining Activation Functions
def sigmoid(Z):
    return 1 / (1 + np.exp(-Z))

def relu(Z):
    return np.maximum(0, Z)

def relu_derivative(Z):
    return (Z > 0).astype(int)

# Step 4: Forward Propagation
def forward_propagation(X, parameters):
    W1, b1, W2, b2 = parameters["W1"], parameters["b1"], parameters["W2"], parameters["b2"]

    Z1 = np.dot(W1, X) + b1
    A1 = relu(Z1)
    Z2 = np.dot(W2, A1) + b2
    A2 = sigmoid(Z2)

    cache = {"Z1": Z1, "A1": A1, "Z2": Z2, "A2": A2}
    return A2, cache

# Step 5: Computing the Cost
def compute_cost(Y, A2):
    m = Y.shape[1]
    # Small epsilon to avoid log(0)
    epsilon = 1e-15
    cost = -np.sum(Y * np.log(A2 + epsilon) + (1 - Y) * np.log(1 - A2 + epsilon)) / m
    return np.squeeze(cost)

# Step 6: Backpropagation
def backward_propagation(X, Y, parameters, cache):
    m = X.shape[1]
    W2 = parameters["W2"]

    dZ2 = cache["A2"] - Y
    dW2 = np.dot(dZ2, cache["A1"].T) / m
    db2 = np.sum(dZ2, axis=1, keepdims=True) / m

    dZ1 = np.dot(W2.T, dZ2) * relu_derivative(cache["Z1"])
    dW1 = np.dot(dZ1, X.T) / m
    db1 = np.sum(dZ1, axis=1, keepdims=True) / m

    grads = {"dW1": dW1, "db1": db1, "dW2": dW2, "db2": db2}
    return grads

# Step 7: Updating Parameters
def update_parameters(parameters, grads, learning_rate):
    for key in parameters.keys():
        parameters[key] -= learning_rate * grads["d" + key]
    return parameters

# Step 8: Training Model
def train_model(X, Y, hidden_size, epochs=5000, lr=0.05):
    input_size = X.shape[0]
    output_size = Y.shape[0]
    parameters = initialize_parameters(input_size, hidden_size, output_size)

    for i in range(epochs):
        A2, cache = forward_propagation(X, parameters)
        cost = compute_cost(Y, A2)
        grads = backward_propagation(X, Y, parameters, cache)
        parameters = update_parameters(parameters, grads, lr)

        if i % 500 == 0:
            print(f"Epoch {i}: Cost = {cost:.6f}")

    return parameters

# --- EKSEKUSI DI GOOGLE COLAB ---

# 1. Load Data (Ganti 'dataset AI.csv' sesuai nama file Anda)
try:
    X, Y = load_and_preprocess_data('dataset AI.csv')
    print("Dataset berhasil dimuat!")
    print(f"Bentuk data Input X: {X.shape} (Fitur, Jumlah Data)")

    # 2. Train Model
    # Input size = 3, Hidden layer = 4, Output = 1
    trained_params = train_model(X, Y, hidden_size=4, epochs=10000, lr=0.1)

    # 3. Prediksi
    A2, _ = forward_propagation(X, trained_params)
    predictions = (A2 > 0.5).astype(int)

    print("\nHasil 10 Prediksi Pertama:")
    print("Target asli: ", Y[0][:10])
    print("Prediksi AI: ", predictions[0][:10])

    accuracy = np.mean(predictions == Y) * 100
    print(f"\nAkurasi Model: {accuracy:.2f}%")

except FileNotFoundError:
    print("Error: File 'dataset AI.csv' tidak ditemukan. Silakan upload file CSV Anda ke Google Colab.")

Dataset berhasil dimuat!
Bentuk data Input X: (3, 100) (Fitur, Jumlah Data)
Epoch 0: Cost = 0.693112
Epoch 500: Cost = 0.668261
Epoch 1000: Cost = 0.437520
Epoch 1500: Cost = 0.266829
Epoch 2000: Cost = 0.191109
Epoch 2500: Cost = 0.148902
Epoch 3000: Cost = 0.123208
Epoch 3500: Cost = 0.106325
Epoch 4000: Cost = 0.094447
Epoch 4500: Cost = 0.085622
Epoch 5000: Cost = 0.078775
Epoch 5500: Cost = 0.073277
Epoch 6000: Cost = 0.068743
Epoch 6500: Cost = 0.064920
Epoch 7000: Cost = 0.061641
Epoch 7500: Cost = 0.058787
Epoch 8000: Cost = 0.056273
Epoch 8500: Cost = 0.054036
Epoch 9000: Cost = 0.052028
Epoch 9500: Cost = 0.050211

Hasil 10 Prediksi Pertama:
Target asli:  [1 1 1 1 1 1 1 1 1 1]
Prediksi AI:  [1 1 1 1 1 1 1 1 1 1]

Akurasi Model: 99.00%
